# Assignment 05: Clean a Lab Export

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

DATA_PATH = Path("data") / "lab_visits.csv"
if not DATA_PATH.exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)
manifest = json.loads(Path("data_manifest.json").read_text())
for name, expected in manifest["files"].items():
    digest = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert digest == expected["sha256"], f"{name} differs from the supplied data: delete data/ and run this cell again"
    print(name, "matches data_manifest.json")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
MISSING_PATH = OUTPUT_DIR / "missing_report.csv"
PROFILE_PATH = OUTPUT_DIR / "typed_profile.csv"
CATEGORY_PATH = OUTPUT_DIR / "category_counts.csv"
VALIDATION_PATH = OUTPUT_DIR / "validation_report.csv"
ROW_COUNTS_PATH = OUTPUT_DIR / "row_counts.csv"
SUMMARY_PATH = OUTPUT_DIR / "clean_summary.csv"

NumPy: 2.3.3
pandas: 3.0.5
data/lab_visits.csv matches data_manifest.json


## Task 1: Find missing values and repeats

### 1.1 Count missing values

Run this cell as it is: it reads the export with no options.

In [2]:
labs = pd.read_csv(DATA_PATH)
rows_read = len(labs)
print("shape:", labs.shape)
print(labs.dtypes)
labs.head()

shape: (60918, 8)
patient_id           str
visit_date           str
clinic               str
gender               str
age                  str
weight_kg        float64
hba1c_pct        float64
glucose_mg_dl    float64
dtype: object


,patient_id,visit_date,clinic,gender,age,weight_kg,hba1c_pct,glucose_mg_dl
0,D10425,2024-01-02,Bayview,M,64,122.7,6.7,164.0
1,D10810,2024-01-02,Mission,F,74,42.0,7.7,172.0
2,D11509,2024-01-02,SUNSET,F,57,70.0,7.0,178.0
3,D13213,2024-01-02,Sunset,F,51,62.3,7.1,172.0
4,D13220,2024-01-02,Bayview,male,61,125.4,5.6,104.0


> **Checkpoint: `output/missing_report.csv`**
> Eight rows under the header `column,missing_count,missing_pct`.

In [3]:
missing_report = pd.DataFrame({
    "missing_count": labs.isna().sum(),
    "missing_pct": (labs.isna().mean() * 100).round(1),
})
missing_report.index.name = "column"
missing_report.to_csv(MISSING_PATH)
missing_report

,missing_count,missing_pct
column,,
patient_id,0,0.0
visit_date,196,0.3
clinic,248,0.4
gender,117,0.2
age,93,0.2
weight_kg,364,0.6
hba1c_pct,4882,8.0
glucose_mg_dl,3741,6.1


### 1.2 Drop no-lab visits and exact repeats

In [4]:
labs = labs.dropna(subset=["hba1c_pct", "glucose_mg_dl"], how="all")

rows_with_labs = len(labs)
print("rows with labs:", rows_with_labs)

exact_repeats = labs.duplicated().sum()
print("exact repeats:", exact_repeats)

labs = labs.drop_duplicates()

rows_unique = len(labs)
print("unique rows:", rows_unique)

rows with labs: 59977
exact repeats: 491
unique rows: 59486


## Task 2: Replace codes and convert types

> **Checkpoint: `output/typed_profile.csv`**
> Five rows under the header `column,dtype,missing_count,min,max,mean`.

In [5]:
labs = labs.replace({"weight_kg": {-999: np.nan}, "glucose_mg_dl": {-999: np.nan}})
labs["age"] = pd.to_numeric(labs["age"], errors="coerce").astype("Int64")
labs["visit_date"] = pd.to_datetime(labs["visit_date"], format="%Y-%m-%d", errors="coerce")

print(labs.dtypes)

typed = labs[["visit_date", "age", "weight_kg", "hba1c_pct", "glucose_mg_dl"]]
typed_profile = pd.DataFrame({
    "dtype": typed.dtypes.astype(str),
    "missing_count": typed.isna().sum(),
    "min": typed.min(),
    "max": typed.max(),
    "mean": typed.mean(numeric_only=True).round(2),
})
typed_profile.index.name = "column"

display(typed_profile)
typed_profile.to_csv(PROFILE_PATH)

patient_id                  str
visit_date       datetime64[us]
clinic                      str
gender                      str
age                       Int64
weight_kg               float64
hba1c_pct               float64
glucose_mg_dl           float64
dtype: object


,dtype,missing_count,min,max,mean
column,,,,,
age,Int64,236,21,94,58.59
glucose_mg_dl,float64,3226,55.0,407.0,145.03
hba1c_pct,float64,3907,3.9,99.0,6.77
visit_date,datetime64[us],299,2024-01-02 00:00:00,2026-06-30 00:00:00,<NA>
weight_kg,float64,940,29.9,1258.0,87.43


## Task 3: Standardize text and encode categories

### 3.1 Clean the text columns

In [6]:
display(labs["clinic"].value_counts(dropna=False))

labs["patient_id"] = labs["patient_id"].str.strip().str.upper()
labs["clinic"] = labs["clinic"].str.strip().str.title().fillna("Unknown")
gender_codes = {
    "f": "F", "female": "F",
    "m": "M", "male": "M",
    "x": "X", "nb": "X", "nonbinary": "X",
}
labs["gender"] = labs["gender"].str.strip().str.lower().map(gender_codes)

display(labs["clinic"].value_counts())
display(labs["gender"].value_counts(dropna=False))

clinic
Mission       18539
Sunset        13976
Excelsior     11739
Bayview        7846
mission        1011
sunset          831
excelsior       681
MISSION         626
SUNSET          445
EXCELSIOR       437
bayview         431
 Mission        417
Mission         410
 Sunset         348
Sunset          337
 Excelsior      290
Excelsior       265
BAYVIEW         260
NaN             238
 Bayview        188
Bayview         170
mission           1
Name: count, dtype: int64

clinic
Mission      21004
Sunset       15937
Excelsior    13412
Bayview       8895
Unknown        238
Name: count, dtype: int64

gender
F      31284
M      26634
X       1231
NaN      337
Name: count, dtype: int64

### 3.2 Bin HbA1c and count the categories

> **Checkpoint: `output/category_counts.csv`**
> Eleven rows under the header `indicator,visits`, from `clinic_Bayview` to `a1c_category_diabetes`.

In [7]:
a1c_bins = [0, 5.6, 6.4, 20]
a1c_labels = ["normal", "prediabetes", "diabetes"]
labs["a1c_category"] = pd.cut(labs["hba1c_pct"], bins=a1c_bins, labels=a1c_labels)

coded = pd.get_dummies(labs[["clinic", "gender", "a1c_category"]], dtype="int64")

category_counts = coded.sum()
category_counts.index.name = "indicator"
category_counts.name = "visits"

display(category_counts)
category_counts.to_csv(CATEGORY_PATH)

indicator
clinic_Bayview               8895
clinic_Excelsior            13412
clinic_Mission              21004
clinic_Sunset               15937
clinic_Unknown                238
gender_F                    31284
gender_M                    26634
gender_X                     1231
a1c_category_normal          8531
a1c_category_prediabetes    17069
a1c_category_diabetes       29918
Name: visits, dtype: int64

## Task 4: Flag outliers and count rule failures

Audit flags identify values to review. Missing gender is flagged but allowed in the cleaned table.

> **Checkpoint: `output/validation_report.csv`**
> Six rows under the header `rule,failing_rows`.

In [8]:
q1 = labs["glucose_mg_dl"].quantile(0.25)
q3 = labs["glucose_mg_dl"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1 {q1}, Q3 {q3}, fences {lower} to {upper}")

labs["glucose_outlier"] = (labs["glucose_mg_dl"] < lower) | (labs["glucose_mg_dl"] > upper)

rules = pd.DataFrame({
    "patient_id_format": labs["patient_id"].str.fullmatch(r"D[0-9]{5}", na=False),
    "visit_date_present": labs["visit_date"].notna(),
    "gender_valid": labs["gender"].isin(["F", "M", "X"]),
    "hba1c_in_range": labs["hba1c_pct"].between(4, 20) | labs["hba1c_pct"].isna(),
    "weight_in_range": labs["weight_kg"].between(30, 250) | labs["weight_kg"].isna(),
    "glucose_within_fences": ~labs["glucose_outlier"],
})

validation_report = (~rules).sum()
validation_report.index.name = "rule"
validation_report.name = "failing_rows"

display(validation_report)
validation_report.to_csv(VALIDATION_PATH)

Q1 119.0, Q3 166.0, fences 48.5 to 236.5


rule
patient_id_format        116
visit_date_present       299
gender_valid             337
hba1c_in_range            62
weight_in_range           47
glucose_within_fences    859
Name: failing_rows, dtype: int64

## Task 5: Clean and summarize

### 5.1 Apply the decisions and count the rows

> **Checkpoint: `output/row_counts.csv`**
> Six rows under the header `step,rows`, from `rows_read` to `weight_blanked`.

In [9]:
clean = labs.copy()
clean = clean[rules["patient_id_format"] & rules["visit_date_present"]]

hba1c_outside = ((clean["hba1c_pct"] < 4) | (clean["hba1c_pct"] > 20)).sum()
weight_outside = ((clean["weight_kg"] < 30) | (clean["weight_kg"] > 250)).sum()
clean["hba1c_pct"] = clean["hba1c_pct"].where(clean["hba1c_pct"].between(4, 20))
clean["weight_kg"] = clean["weight_kg"].where(clean["weight_kg"].between(30, 250))
clean["a1c_category"] = pd.cut(clean["hba1c_pct"], bins=a1c_bins, labels=a1c_labels)
# Keep category_counts.csv as the Task 3 snapshot
clean = clean.reset_index(drop=True)

row_counts = pd.Series({
    "rows_read": rows_read,
    "rows_with_labs": rows_with_labs,
    "rows_unique": rows_unique,
    "visits_dropped": len(labs) - len(clean),
    "hba1c_blanked": hba1c_outside,
    "weight_blanked": weight_outside,
}, name="rows")
row_counts.index.name = "step"

print("shape:", clean.shape)
display(row_counts)
row_counts.to_csv(ROW_COUNTS_PATH)

shape: (59071, 10)


step
rows_read         60918
rows_with_labs    59977
rows_unique       59486
visits_dropped      415
hba1c_blanked        61
weight_blanked       46
Name: rows, dtype: int64

### 5.2 Validate, then save the summary

> **Checkpoint: `output/clean_summary.csv`**
> Four rows under the header `,count,mean,std,min,25%,50%,75%,max`.

In [10]:
checks = pd.Series({
    "one row per visit": clean.duplicated(subset=["patient_id", "visit_date"]).sum() == 0,
    "patient IDs valid": clean["patient_id"].str.fullmatch(r"D[0-9]{5}").all(),
    "visit dates present": clean["visit_date"].notna().all(),
    "hba1c in range": clean["hba1c_pct"].dropna().between(4, 20).all(),
    "weight in range": clean["weight_kg"].dropna().between(30, 250).all(),
})
display(checks)
assert checks.all(), checks[~checks]

clean_summary = clean[["age", "weight_kg", "hba1c_pct", "glucose_mg_dl"]].describe().round(2).T
display(clean_summary)
clean_summary.to_csv(SUMMARY_PATH)

one row per visit      True
patient IDs valid      True
visit dates present    True
hba1c in range         True
weight in range        True
dtype: bool

,count,mean,std,min,25%,50%,75%,max
age,58836.0,58.59,12.81,21.0,50.0,59.0,68.0,94.0
weight_kg,58096.0,86.84,19.27,30.0,72.9,85.4,99.5,250.0
hba1c_pct,55123.0,6.7,1.08,4.0,5.9,6.6,7.4,20.0
glucose_mg_dl,55867.0,145.03,35.82,55.0,119.0,141.0,166.0,407.0


### 5.3 Save the cleaned table as Parquet

> **Checkpoint: `data/labs_clean.parquet`**
> Not committed or checked: `data/` stays out of Git.

In [11]:
parquet_path = Path("data") / "labs_clean.parquet"
clean.to_parquet(parquet_path)

clean_back = pd.read_parquet(parquet_path)
print(clean_back.dtypes)
print("same as clean:", clean_back.equals(clean))

patient_id                    str
visit_date         datetime64[us]
clinic                        str
gender                        str
age                         Int64
weight_kg                 float64
hba1c_pct                 float64
glucose_mg_dl             float64
a1c_category             category
glucose_outlier              bool
dtype: object
same as clean: True


## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [12]:
from check_assignment import run_checks

_ = run_checks()

Checks: used the bundled copy because the latest could not be fetched (URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>).
[PASS]  4/4  missing report: one row per column
[PASS]  5/5  missing report: missing_count
[PASS]  4/4  missing report: missing_pct
[PASS]  4/4  typed profile: -999 read as missing
[PASS]  4/4  typed profile: age as numbers
[PASS]  3/3  typed profile: unreadable ages left blank
[PASS]  3/3  typed profile: visit dates parsed
[PASS]  2/2  typed profile: undated visits kept
[PASS]  3/3  typed profile: other values unchanged
[PASS]  4/4  category counts: clinic names cleaned
[PASS]  3/3  category counts: missing clinic filled
[PASS]  4/4  category counts: gender codes mapped
[PASS]  5/5  category counts: a1c_category
[PASS]  3/3  category counts: indicator columns
[PASS]  3/3  validation report: one row per rule
[PASS]  2/2  validation report: rule names
[PASS]  4/4  validation rep